# Plain Llama text for the authorship classifier (DATA 298B, PUK-28)

**What this does.** Writes about 900 short pieces with plain Llama 3.1 8B Instruct (no fine-tuning, no retrieval) on
450 Lemkin blog titles that are NOT among our 30 gold topics. The classifier uses this text as its "plain LLM writing"
negatives, so none of our systems' drafts are needed for training. It uses the same prompt wording and settings as our
Base Llama baseline (temperature 0.7, top-p 0.9, repetition penalty 1.15, limits 768 / 320 / 768 tokens).

**How to run.** Runtime > Change runtime type > GPU (A100 if you have it, otherwise it falls back to 4-bit on a smaller
GPU, which is slower). Then Runtime > Run all. Nothing to upload and no Hugging Face login needed.

**How long.** About 30 to 60 minutes on an A100 (about 10 minutes of that is the model download). Much longer on a T4.
If Colab disconnects, run all again: finished items are skipped.

**What to send back.** The last cell downloads `llm_negatives_bundle.zip`. Put it in the `authorship_classifier`
folder on the laptop and tell Claude. Nothing private is in it: it is Llama writing, not Lemkin's posts.

In [1]:
!pip -q install -U "transformers>=4.44" accelerate bitsandbytes

In [2]:
import collections, hashlib, json, os, platform, re, statistics, time, zipfile

MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct"   # open copy of Meta Llama 3.1 8B Instruct, no login needed
SEED = 42
OUT_DIR = "/content/llm_negatives"
os.makedirs(OUT_DIR, exist_ok=True)
JSONL = f"{OUT_DIR}/llm_negatives.jsonl"

# Same wording as the Base Llama baseline (experiment EXP-20261001-010)
INSTRUCTION = {
    "blog": "Write a blog post in the style of Jason Lemkin about: {topic}",
    "linkedin": "Write a LinkedIn post in the style of Jason Lemkin about: {topic}",
    "talk": "Write a talk in the style of Jason Lemkin about: {topic}",
}
MAX_NEW = {"blog": 768, "linkedin": 320, "talk": 768}
SAMPLING = dict(do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.15)
# which slice of TOPICS each medium uses (blog and talk share the first 150 topics, LinkedIn uses the rest)
PLAN = {"blog": (0, 450), "talk": (0, 150), "linkedin": (150, 450)}

### Topics
450 titles of real Lemkin blog posts from the training split. Removed: the 30 gold topics, their near copies and families,
event promos, and duplicates. The list is built into this notebook, so there is nothing to upload.

In [3]:
TOPICS = json.loads(r'''["Don’t Fear Business Process Change. It’s (Usually) a Good Thing.", "Do Your VPs Have a Mentor Budget?", "Be Honest If Growth Has Slowed. Is Your Product Still Magical? It Probably … Used to Be.", "The Top Product Roadblocks and Headaches Scaling from $0 to the First $100M ARR with Neo4j CTO Philip Rathle", "Thinking of Being a VP at a Venture-Backed Startup? Think Twice", "8 Ideas to Help Make Sure Your Team … Stays", "Raising Money After a Rough Patch? At Least Put Up 3 Good Months in a Row", "5 Interesting Learnings from Toast at $500m in ARR, and a $3B Total Run Rate", "By Late 2025, Replit Got Really Good. Now Imagine If Your Agents Really Could Run 24×7.", "What Are The Odds You Get Acquired Within 5 Years for a Good Price? Around 1%-1.5%", "How To Know You’ve Hit Initial Traction In SaaS. The Moment When You’ve Got A Real Company.", "Who Could You Buy With 10% of Your Market Cap?", "An Underused Strategy: Truly High Value Drip Marketing to Prospects", "Some Folks in SaaS Are Still Doing Really, Really Well. E.g., MongoDB, HashiCorp, Monday, GitLab, Okta and More.", "Why Every B2B Software Company is About to Become a Backend Service. You Need To Get Ahead Of It.", "No, Inbound Isn’t Dead. The GTM Playbook Isn’t Broken. But Your Moats Are Shrinking to Months. SaaStr AI London AMA LIVE with Jason Lemkin", "Champion Change: You Gotta Jump On It", "Don’t Confuse Room at the Bottom with Disruption", "How Many Sales Reps Do I Need To Hit The Plan For The Year? Hint: ~2x What You Think (Updated)", "Your VCs May Be Giving Up on You Right About Now. Don’t Let That Stop You.", "The 8 Things No One Tells You About ‘Prosumer’ Vibe Coding (That You Need To Learn To Build a Real B2B App)", "Hiring Your VPs: When Can You Compromise? (UPDATED)", "7 Thoughts on Building Your First Partner Program", "The $400B HR Tech Boom: How Old School ADP & Paychex Are Thriving Alongside Rippling, Deel & Gusto", "The Future of B2B Marketing is AI Agent Recommendations. And AI Agents Play Favorites.", "How Top SaaS Companies Drive 2-3x Better Campaign Performance Through Multi-Channel Personalization with Customer.io", "Sam Blond + Jason Lemkin “GTM in 2025: How It’s Changed, How It’s Changing, and What Hasn’t Changed (Yet)”", "The Top 10 Videos of the Week: ClickUp, Y Combinator, Unsplash, ZoomInfo and More!", "You’re Lucky if Even 10% of Your Potential Customers are “In Market” At Any Given Time", "Scaling to $3B: How Clio Built a Vertical SaaS Powerhouse with AI and Innovation with CEO Jack Newton", "My App Stack: Carin van Vuuren, CMO of Greenhouse", "The Last 4 “SaaS Crashes” Barely Made a Dent in the Boomtimes", "Cutting Off Your Long Tail: When Is It Worth It?", "What To Look For In Your First Marketing Hire", "Assume the Customer Knows Everything", "Your Job In AI + B2B in 2026: Grab The Tailwinds, For Real. It’s Not Too Late. But It’s Different. (Updated with Video)", "10 Tips To Getting Better at Hiring in the Earlier Days", "The Two Things to Do. When You Don’t Know What to Do. (Updated)", "Dear SaaStr: How Do I Know If I Have Product-Market Fit for Real in the Early Days?", "The 2026 SaaS Crash: It’s Not What You Think", "What a Typical Day Looks Like for a SaaS Sales Rep", "The 10x Rule: What Raising $1 of Venture Capital Really Means", "A Low Win Rate is Just a Huge Opportunity-in-the-Making", "The Top 10 Worst Pieces of SaaS Advice (Updated)", "Dear SaaStr: How Do I Manage a Partner Referral Process?", "The Top 5 Mistakes Founders Make With Their VC Investors", "10 Ways to Build a Moat in SaaS. But AI is Also Making Them Weaker.", "Customer Success Is A Single Digit Hire (Updated)", "10 Things That Would Have Helped Me Go From $1m to $10m Faster with Less Stress", "What A New VP of Sales Needs To Be Doing Their First 2 Weeks", "The Honest Reason AI Will Beat Many Sales Reps: They Don’t Really Do A Lot Of The Work. But Agents Do and Will.", "The CEO Doesn’t Have to Code. But They Do Have to Be Great at One Core Function.", "The Single Digit Growth Club: One With Too Many New Members", "Dear SaaStr: What’s The Best Way to Connect with VCs?", "VCs Don’t Really Ghost You. They are Just Prospects. Treat Them as Such.", "Venture Backed or Bootstrapped? There’s a Third Way. Just Raise One Round.", "SaaStr AI 2026 is Running 132% of Last Year. But It’s Not Remotely That Simple. It Could Have Been -46%", "Right Now, Managing AI Agents is About as Much Work as Managing Humans. Just Different Work.", "Dear SaaStr: How Do I Become a Great Sales Rep?", "From Zero to 20 AI Agents in 10 Months: The SaaStr Playbook for Actually Deploying AI Agents That Work", "Q1 is Your Last Chance to Get the Hires Done You Need for 2022", "At the Top SaaS Companies, Founder-CEOs Own ~15% at IPO. And Most Co-Founders Are Not Equal (And That’s OK). UPDATED", "Dear SaaStr: Why Do Big Companies Buy Small Companies Instead of Competing?", "ElevenLabs: From 0 to $300M ARR in 3 Years. Why The Best AI B2B Apps Are Exploding", "Dear SaaStr: I Can Do My Sales Job in 15 Hours a Week. Should I Take a Second Job?", "The Top 10 Pieces of Advice I’d Give to My Younger CEO Self", "Dear SaaStr: When Do “Buy Out” Contracts Make Sense?", "Dear SaaStr: A VC Seems Interested. What Are the Odds a Deal Actually Happens?", "The 10x Feature is Real. At Least, for a While. What’s Yours?", "Building and Scaling an AI Company in 2025: What Top VCs Really Want to See with Zetta, Glasswing and B Capital", "Why Vertical SaaS is Booming, And How to Get 110% NRR from SMBs, with Jason Lemkin + Mangomint’s CEO Daniel Lang", "From Zero to $25B in 17 Years: The Inside Story of HubSpot’s Domination Strategy", "5 Interesting Learnings from Zendesk. As It Crosses $1B in ARR.", "Dear SaaStr: How Do You Know When It’s Time to Sell Your Startup?", "Dear SaaStr: How Can a SaaS CEO Tell Which Customers are 10+ Year Customers?", "Canva Is Giving Its 5,000 Employees a Week to Learn AI. Will It Work?", "If You’re Behind on AI, Put Your 10-20 Best Engineers in a Different Building. Today.", "To Thrive Today, You Have to Become An Agentic Deployment Expert. But So, So Few Actually Are.", "What’s Actually Working For SaaS PR In 2023 … And What Isn’t", "Dear SaaStr: Should I Switch from HubSpot CRM to Salesforce if I’m Just Starting Out?", "A 10 Point Checklist for Hiring Your First Few Sales Reps", "How Hard It Is to Be a Great VC? Vencap’s Real Hard Numbers from 1,900+ Funds", "Our 20+ AI Agents and Their Moats: Real But Weak", "Dear SaaStr: What “Quotas” Should My VP of Customer Success and VP of Product Have?", "The Real Question for Outcome-Based Pricing Is If It Will Stick", "Look, No VC Wants to Fund a Startup With So-So Growth. Except Maybe Your Existing Investors.", "85% of Business App Trials Start on Day 0. Don’t Waste Your Shot. The Latest Data from 75,000 Apps and RevenueCat", "Loveable and Replit Both Hit $100M ARR in Record Time. The Vibe Coding TAM: How Big Can This Market Really Get?", "Dear SaaStr: Why Do Angels Invest Even if Most Of Them Fail? And is a 10x Return Even Enough?", "Dear SaaStr: How Do You Stay Calm as a CEO in Tougher Times? Or Even Just in General?", "The One Secret Tip To Hiring a Great Sales Rep (In The Early Days)", "A Checklist: 10 Signs You Shouldn’t Hire a Sales Rep", "It Was an IPO-a-Day in 2021. We’ve Never Seen That Before", "The $10M to $1B Vertical SaaS Playbook: Key Lessons from Procore’s Chief Product Officer Wyatt Jenkins", "The Labor Gap No One Talks About: Why AI Will Fill the Jobs Humans Won’t Do in Tech", "Where Are The 2021 Unicorns Today? 60% Are Stuck In Limbo, Per Carta", "8 Signs a New VP Won’t Work Out. And 3 Things Not to Worry So Much About.", "Is SaaS Dead? No. But One Thing Is Clear: It’s Unstable.", "There Are Never Enough Stock Options To Go Around", "The Bar Today for a Series B", "Dear SaaStr: How Does an Investor Protect Themselves When Investing in a Startup?", "10 Things My VP of Marketing Did — and Didn’t — Do in Their First 30 Days", "Dear SaaStr: How Do I Know If It’s The Right Time to Sell My Company?", "7 Things The Best Sales Reps Get Right. And 10 Ways The Rest Make Things Worse.", "What Will Stay Different After Covid", "Dear SaaStr: What’s a “Messy Cap Table?” And How Do I Fix It?", "10+ Signs of a Mediocre Hire", "Salesforce, Microsoft, Google and Atlassian All Raise Prices Again in 2025. Hooray.", "10 Tips to Make Your VCs Calmer, Happier — And On Your Side", "Dear SaaStr: How Much Do Founder-CEOs Own at Time of Exit?", "10 Great Learnings on How to Scale Selling to SMBs from Yamini Rangan, CEO of HubSpot", "How 1,000+ B2B Startups Are — And Aren’t — Growing. The Real Data.", "The $8 Billion+ B2B Marketing Strategy That Loses 10% Revenue in Month 1 (But Creates Unbreakable Customer Loyalty) with RevenueCat’s VP of Growth", "We Thought These Leads Were Dead. Our AI Proved Us Wrong (And Booked 4 Meetings in 48 Hours)", "Dear SaaStr: Can A Great Sales Rep Sell Any SaaS Product?", "It’s Never Too Late to Go Upmarket (See, E.g., PagerDuty, RingCentral, Asana, etc.)", "AI Agents Catching Other AI Agents Cutting Corners and Hallucinating. And Why That Means AI is Getting So Much Better", "A Soft Miss vs. A Hard Miss in Sales (Updated)", "It Takes a Decade in SaaS to Go Big: Can You and Your CoFounder Do The Time?", "How To (Successfully) Source Your First 2 Sales Reps", "The Real Data on What it Takes to Go Big and Eventually IPO with Meritech Capital", "Don’t Hide in Zero Cost Marketing", "Dear SaaStr: What Are SaaS Billionaires Like?", "An 8 Part Test to Identify Bad Investors Before Signing a Term Sheet", "VC Funding Hit a 5+ Year Low in Q1’23: CBInsights and Carta", "Nvidia Created 27,000 Millionaires Already. What It Teaches Us About Getting Rich. And The Power of Going Long.", "KeyBanc: Only 41% of SaaS Companies Price By Seat Now", "No, There Aren’t That Many Deals Where the VCs Make Millions and The Founders Make Nothing", "Segmenting Your Sales Goals by Customer Category: Growing. Struggling. And Dire Straits", "AI Agents in B2B: Top 10 Learnings from Aaron Levie, CEO of Box and IBM’s VP of AI", "We’re Literally Just Getting Started in AI + B2B. Here’s Why 2026 Is When It Gets Real.", "The Playbook for Going Upmarket with Stripe’s CBO and Checkr’s COO", "Dear SaaStr: Why Is a “Downround” So Bad?", "The 2026 Sales Team: 50% Humans, 50% AIs. All 1 Team.", "The Brutish Sales Playbook Too Many Are Running Today", "The Advice That Stings is The Advice You Want to Hear", "Emergence: Top Quartile Startups Are Growing 58% at $10m in ARR", "9 Quick Upgrades to Run Your Company Better. Right Now.", "Databricks at $5.4B, Growing 65%. Is It the Most Unstoppable Company in B2B Right Now?", "The Job of a VC: Turn Every $50m into a $2B+ Exit", "10 Tips To Avoid SaaS Burnout", "Dear SaaStr: We’re Only at $3m ARR After 11 Years. Should We Keep Going?", "3 Big Changes in Running SaaS Companies in 2024", "6 Months of AI SDRs: What’s Worked, How They Brought In $1M+ in 90 Days, and the Real Data Everyone’s Asking For", "Your Product Has to Be Easier to Buy Than to Use", "What’s Your Loss Rate? You Really Should Know", "Everything’s In Play in the Age of AI: Why Only 1 of Our 16 Core AI Agents Comes From a Legacy Vendor", "Every Top Founder Ever: “I Should Have Acted on Bad Trends Earlier”", "The Early Days: 5 Things Vanta Got Right, And 5 It Got Wrong, Getting to First $10m ARR", "Dear SaaStr: What is RevOps Responsible For in a B2B Company?", "3 Basic Steps to Reduce Sales Rep Turnover", "15+ of The Top Sales & Marketing Mistakes B2B Startups Make", "Yes, You Need to Focus. But in 2025, Step Up or Step Aside.", "Dear SaaStr: How Common is it for Cofounders to Have Equal Splits in Terms of Equity?", "10 Things To Know About Raising Venture Capital for The First Time", "Dear SaaStr: How Much of a Cash Buffer Should I Have For a “Rainy” Patch?", "SaaStr’s AI Agent Playbook: How We Deployed 20+ Agents to Scale 8-Figure Revenue with Single-Digit Headcount", "When Your Founders Turn Down Nine Figures: The Beast Mode 2.0 Framework", "Dear SaaStr: What Are Some Signs a StartUp Team is Going to Underperform or Overperform?", "That First Magical Sales Rep (Updated)", "Doubling Down: Harpinder Singh, Partner at Innovation Endeavors", "Why Marketing Should Almost Never Report to Sales", "10+ of The Top Mistakes Sales Reps Make … That Kill Deals", "FINAL SaaStr Scale Session This Week: Therese Tucker, CEO/Founder of BlackLine!", "The Top 10 SaaStr 2025 Sessions So Far! OpenAI Head of GTM, HubSpot CEO, Canva CCO, DropBox and Snowflake CEOs!", "20 More of the Top SaaStr Surveys on Hiring, Support, Customer Demos, Turn-Over, and More!", "Top SaaStr Posts and Pods of the Week: Wiz and Rippling’s CROs; Google Cloud’s VP of Marketing; And Great Dashboards", "5 Ways to Mitigate Stress In Your SaaS Start-Up", "10 Things That Are Different in Vertical SMB Sales with Toast’s CRO", "The Beauty and ROI in The Zero Voluntary Attrition Sales Team", "Sales Often Really Doesn’t Understand Customer Success", "4 Reasons to Send Investor Updates Even When Times Are Tough", "Shopify’s $10.7B ARR vs. HubSpot’s $3.1B vs. Box’s $1.1B: How Three 2014-2015 IPOs Delivered Wildly Different Revenue Outcomes", "ChartMogul: SaaS Growth Rates Peaked in Q1’21 — And Have Fallen Ever Since. But They Did Rebound A Smidge in 2023.", "10 Things That Tell a VC You May Not Be Ready for “Prime Time”", "How to Sell a Second Product", "Dear SaaStr: What’s the Best Way to Help Top SDRs Become SMB Account Execs?", "The Latest B2B IPO is Netskope. And it’s Fire. $700m ARR, Growing 33%. The Next IPOs Will Be Even Better.", "Salesforce Grows a Stunning 23% at $24B in ARR. 5 Interesting Lessons.", "From $30M to $11B: The ServiceTitan Playbook – CRO Ross Biestman’s Masterclass on Vertical SaaS", "13 Things That Say “Don’t Fund Me” To VCs", "Reinvent the Wheel on Product. Not on Pricing.", "Next Time — Ask For $1,000,000. A Year. I’m Not Kidding.", "Dear SaaStr: We Have a Ton of VC Interest But Are Profitable. Should We Still Raise?", "A Few VP of Sales / CRO Personas to Stay Far Away From", "One Simple Rule On How Much To Pay Yourself Once You Raise Venture Capital", "All Your VPs Really Need to Do is Tilt the Curve", "10 Months Ago, We Were Barely Using Salesforce. Now It’s Our AI Agent Hub.", "8 Simple Ideas to Inspire The Team", "Where ‘Prosumer’ Vibe Coding Falls Short Today: Security. It’s The #1 Reason “Roll Your Own” Isn’t Prime Time Ready", "Dear SaaStr: When Should I Hire My First AE?", "Dear SaaStr: When Do You Start to Develop a Brand in SaaS?", "Why a Lot of SaaS is Still “Terrible”: You Can’t Just Talk to It and Have It Do What You Want. It’s Time in the Age of AI.", "How A16Z’s Returns Show Just What an Epic Year and Epic Time 2021 Was. Will 2028 Be Next?", "30+ SaaS CEOs Share The Signs Their First VP of Sales … Just Wasn’t Going to Work Out (Updated)", "The Challenge with SMB SaaS: High Growth Can Only Mask High Churn For Just So Long", "How Our AI BDR Drove Inbound From 30% to 71% of Revenue from Inbound — By Letting AI Qualify 24/7", "69% Of You Are Seeing Demand Pick Up — At Least a Little Bit", "Dear SaaStr: How Should I Actually Set the Sales Plan for the Year For My VP Sales?", "Vendr: Renewals Are Consuming More and More of IT Budgets. 80%+ Now.", "2 Simple Edges Anyone Can Get in Sales. And Pull Ahead of the Pack.", "The Present and Future of AI in Sales and GTM A Deep Dive with Jason Lemkin and Kyle Norton, CRO at Owner", "Dear SaaStr: What Are The First Things VCs Notice During a Pitch?", "Who Should Marketing Report To? What 3,000+ Companies Actually Do (And Why It Matters)", "Dear SaaStr: How Much Money Do The Best SaaS Inside Sales AEs Make?", "Dear SaaStr: How Many Employees Does a SaaS Company Have at 100M ARR?", "Why You Don’t Get Any Credit Anymore for 30% of Your Code Being Written By AI", "When an Investor Puts Their Career On The Line For You", "It’s Time for a New VP. Can You Pull Off a Stealth Search?", "Dear SaaStr: How Should Startup Founders Approach Early-Stage Hiring?", "Dear SaaStr: I’m Starting as a New VP of Sales. How Can I Add Value Quickly?", "In Startups, The Harder You Work in Sales, The More Good Leads You Get", "The 10 Most Popular SaaStr Videos Of All Time: CEOs YCombinator, WordPress, GitLab, Loom and More!", "The $939B Question: Is AI Eating SaaS or Feeding It?", "The “Headless” Sales Team Never Really Works", "The Top 10 Customer Success Metrics Investors Really Care About in 2025 with Gainsight’s CEO Nick Mehta", "SaaS Is Still Slowing Down, Unfortunately: What Q1 2025 Numbers Reveal About the Cloud Software Market", "The Growth Tactics That Worked For Me in Early Days. We’re All Different, Though. (Updated)", "The Top 10 Mistakes Founders Make After $10m ARR", "The Keys to Getting to $500M ARR In Just 7 Years with Attentive’s CEO", "Top 10 SaaStr AI Predictions for 2026", "Events Work. Events Are Back. But Are They Worth the Big Expense?", "Why Only “WTF” Products Can Survive Today with Brett Queener Partner at Bonfire Ventures", "Dear SaaStr: What Is It Like to be Pitched by Startups as an Investor?", "What Order Should You Hire Your Management Team In?", "5-6 Years Is About As Fast As You Can Mint a True VP of Sales", "Top 10 Interesting Learnings From Figma’s IPO … That You May Have Missed", "10 Tips to Fundraising in The Tumultuous Times of 2023", "Dear SaaStr: I Want to Roll Out an AI SDR. How Do I Start?", "34% of All Startup Acquisitions Are By Other Start-Ups. A New Record.", "Dear SaaStr: How Are VP of Sales Commissions Normally Structured?", "7 Tips to Close 7,000 Customers From Gorgias’ VP of Sales", "Dear SaaStr: Should I Take On Venture Debt?", "The Sales Playbook for 2022: A Deep Dive with Brendon Cassidy + Jason Lemkin", "Freemium is Back — The AI Edition! But You’ll Still Probably Need 50 Million Active Users for Freemium to Actually Work as a Business Model.", "The 48 Types of VP Sales. Make Deadly Sure You Hire the Right One.", "Building AI that Actually Works: 10 Key Lessons from Calendly’s CPO and Head of UX", "Dear SaaStr: What Was Your First Meeting Pitching a VC Like?", "The Top 10 Important Finance Mistakes First Time Founders Make", "From 0 to 1 Billion Meetings: How Otter.ai Built a Bottom-Up AI SaaS Without a Single Sales Rep", "10 Awesome Things the Free SaaStr AI Can Do For You, From Reviewing VC Pitches to Drafting Cold Outbound For You", "CEO Compensation Today: Is It Broken? A Deep Dive With Brian Halligan Chair of HubSpot on What’s Wrong and How to Fix It", "When You Want to Hire a VP From a Big Tech Company, What Are You Competing With? $800,000 a Year", "SVB’s State of the Markets Report: AI Isn’t More Efficient, VC Dollars Are Even More Concentrated, and M&A is Picking Up But Fewer Deals Make Money", "It’s Just Better If Your VCs All Trust Each Other (Updated)", "Zuora Goes Private for $1.7 Billion. Why This is Good News for SaaS.", "Dear SaaStr: What Are Your Top Tips For Hiring Our First Few Sales Reps?", "Competition In SaaS Leads to A Lot of Things. But Not Lower Prices, Usually.", "The 10% Rule: How Much Should You Really Invest in Customer Success? With the CRO of Notion and VP of CS at GitHub", "From 0 to $10m ARR, At What Point Do We Start Hiring and Whom?", "7 Simple Ideas to Immediately Improve Your Conversion Rates in SaaS", "Dear SaaStr: As a Co-Founder, When Do You Know It’s Time to Quit?", "Top 10 SaaStr Videos of the Week: Algolia, Stripe, Intercom, Qualtrics and More!", "Dear SaaStr: What Metrics Should I Be Using to Measure My Outbound Team?", "Dear SaaStr: What Should I Look For In My Very First Sales Rep?", "The Richest Person Battle: What Larry Ellison’s Rise to #1 Tells Us About AI, Enterprise Software, and Why Revenue Still Matters Most", "Iconiq’s State of Software in 2025: Forward Deployed Engineers Up 12x; 55% of GTM is Now in Post-Sales; 2025 Is Crushing 2024 For Funding", "FREE Upcoming Workshop Wednesdays with A16Z, Notion, ex-CMO Calendly, Atomic, Weights & Biases, ex-CRO Divvy and More!!", "Dear SaaStr: I Don’t Really Feel Confident Enough to Call Myself “CEO”", "The Silent, Lurking Churn: Activation Rates Less Than 90%", "Let’s Be Clear, 2021 Was Insane", "The First $100,000,000 ARR at Ramp: How CEO Eric Glyman and CTO Karim Atiyah Built a $30B+ Finance Platform Through Asymmetric Bets", "How Vercel Hit $9.3B and Replit Hit $3B… After a Decade. The Long Paths to AI “Overnight Success”", "“What If I’m Not Sure that My VP Sales is The Right Person?”: An Honest 5 Point Test", "SaaS at Scale: Hard-Won Lessons from Cliff Obrecht on Building Canva from $0 to $4B ARR", "A Simple 4 Part Test to Know If You’ve Hired the Right VP", "10 Things to Know Before You Deploy Your First AI SDR: The Very Latest with SaaStr’s Jason and Amelia", "Times Are Tough for Some. But — 25+ of The Leading SaaS and Cloud Companies Are Up 69% on Average This Year", "AI is Already Increasing Our Cognitive Load. That’s Only Going to Increase — Dramatically.", "How to Build a World-Class Customer Success Machine: Top Lessons from the CRO of Notion and VP of CS at GitHub", "Dear SaaStr: I Just Sent an Email to a VC With Two Typos. Should I Resend the Email?", "How to Steal a Customer From the Competition", "If You Are Going to Hire a Stretch VP — Do It Early.", "Thoughts on How to Get The Team To Work “Harder”. Well, Really — Better.", "Wix: The Unexpected New Threat to Replit and Lovable? A Deep Dive into How Its Vibe Coding App is Now Growing Just As Fast, Or Close To It", "From $10M to $100M ARR in 5.5 Months: Inside Replit’s AI Coding Rocketship", "Get EXCITED!! The First SaaStr.ai Summit is Sep 10-12 in SF Bay!!", "Doubling Down: Peter Specht, Partner at Creandum", "10 Crystal Clear Signs Your VP of Sales Just Isn’t Going to Work Out", "50 Top Tips To Take Friction Out of Your Sales Processes. From the CROs of Rippling, Brex, Slice, Grafana Labs and More", "AI Adoption: 5 Hard Truths About the Fastest Technology Transformation in History With Aaron Levie, CEO Box", "5 Nonobvious Learnings from Atlassian’s Path to The First $10B in ARR With Ex-CRO Cameron Deatsch", "We Deployed 20+ AI Agents and Replaced Our Entire Human SDR Team. Here’s What Actually Works. (Video + Pod)", "Your Top 5% Employees … And Bottom 5%. It’s Time to Go Through Them.", "The Best SaaS Founders Are (Somehow) Indefatigable", "Dear SaaStr: What’s The Typical Enterprise SaaS Cancellation Policy For $100k ACV/Bigger Deals? Should The Contract Have a Fine?", "If the VP of Sales Doesn’t Believe, It’s All Over", "SaaStr AI App of the Week: Lightfield — The AI-Native CRM That Updates Itself, Preps Your Meetings, and Actually Knows Your Customers Cold", "7 Pieces of Advice Entrepreneurs Never Hear. That They Need to Hear. (Updated With Video!)", "Top 10 SaaStr Videos of the Week: Twilio, MongoDB, Blackline, Stripe and More", "The Private Equity Math of 1+1=3 in SaaS Acquisitions", "Dear SaaStr: Who Drafts the Term Sheet, The Entrepreneurs or the VC?", "Customers Are Asking for Longer Payment Terms. Should You Push Back?", "The $4.5B Board Meeting: Inside Snowflake’s AI Transformation and the CEO Who Bets His Company on Their Platform", "5 Pros, and 5 Cons, to Taking a VC Round at a Very High Valuation", "Have Even Half-Decent Connect Rates? Double Down on Outbound Sales Early On", "Business Disputes: You Always Lose", "The Great IPO Awakening: Are We Back?", "Who’s Left In the 10x ARR Club? The Incredible Shrinking Elite of Public B2B Companies Still Trading at 10x+ Revenue. But … The Best Is Yet To Come.", "Dear SaaStr: I’ve Bootstrapped to $3m ARR with Founder-Led Sales. Am I Ready to Hire a Full Time VP Sales?", "10 Learnings from Anjali Sud, CEO of Vimeo on How to Do Self-Service and Enterprise Right", "Everyone in SaaS Needs to Do Customer Support. At Least Until You Have 50-100 Employees. But Ideally, Forever.", "Doubling Down: Lucy Deland, Co-founder & General Partner at Inspired Capital", "My App Stack: Neeracha Taychakhoonavudh, Global Head of Customer Experience at Asana", "Dear SaaStr: Under What Circumstances Should the Board Fire the CEO?", "Ask Every VP You Interview for A List of Everything They Want to Do. They’ll Only Do The Top 3-4.", "Are You Really On The IPO Track? If Not, Maybe Don’t Unicorn", "Scaling 6 Products to $100M+ ARR Each: Samsara’s CPO Kiren Sekar on Multi-Product Growth", "The Capital Everyone is Regretting Right Now is the “Free” Capital", "The Ultimate SaaStr Sales & GTM Reading List: 30 Essential Articles Every Revenue Leader Must Read", "How OpenAI Built a Global CS Organization in 18 Months: Vanessa Gatihi’s Playbook for AI-Powered Customer Success", "Meritech’s Look at 2025 Exits: M&A at $587B Hits a Decade+ High … But IPOs Were a Whimper", "Perplexity’s CBO: Speed is The New Moat in AI Sales", "5 Changes You Can Make >Today< To Grow Faster", "Dear SaaStr: Do Companies Prefer Stock Over Cash When They Acquire Startups?", "In The Early Days, What You Think is a “Sales” Problem is More Likely a Marketing Problem (Updated)", "Dear SaaStr: How Can I Earn a Price Increase?", "You Can Hide From a Lot of Things as a Founder. Cash Burn Isn’t One of Them.", "My App Stack: Luca Penati, CMO & CCO of BEE", "The Early Days: How Deel Went From $1m to $100m ARR in Just 20 Months", "Dear SaaStr: What Does a Day in the Life of a SaaS Founder Look Like Post Launch?", "Jason Calacanis + Jason Lemkin on Slack, Zoom, Covid and the Cloud, and More", "The 3 Financial Plans You Need for The Year: C-90, C-60 and C-10 (Updated)", "SEM Rush: If You Get In The “Top 10” For SEO, You Probably Will Be There in AI Search, Too. But — Differently.", "Dear SaaStr: How Do You Respond to a VC Who Says They Don’t Invest in Your Space?", "Stop Promoting Your Startup. Start Promoting Real Value in Your Industry.", "It Can Take Years to Get Acquired For Millions. So Start Nurturing Those Relationships Now.", "How Nebius Built a $4B+ GPU Cloud in 18 Months: The 7 Top Lessons Learned", "12 Of The Most Common Mistakes Made Building Your First Sales Team", "11 Things That Set The Best Sales Teams Apart From The Rest", "AI in Sales: Incredible When It Works. And Often a Net Negative When You Use It On Autopilot", "How To Transition from Founder-Led Sales: Why Most Get It Wrong and How to Get It Right", "5 Tips For Pitching VCs for The Very First TIme", "The Confounding Logic of Discounting", "Doubling Down: Hilarie Koplow-McAdams, Venture Partner at NEA", "The Cutting Edge in Customer Success: 7 Learnings With Nick Mehta and Jason Lemkin", "10 Things That Always Work in B2B Marketing", "Dear SaaStr: We Just Raised a VC Round. Now Everyone Wants Big Raises. What Do I Do?", "My Top 10 Year One SaaS Mistakes. Save Yourself Some Pain & Just Don’t Make Them Yourself.", "AI Apps The SaaStr Team Loves — and Uses Every Day", "6 Types of Bad Behavior Sales Reps Bring Out When They Are Struggling to Hit the Plan", "Top SaaStr Posts, Vids and Pods of The Week: YCombinator, CEO WordPress, CEO ZoomInfo, CRO Anthropic and More!!", "9 Thoughts on What Makes For a Good Pricing Page (UPDATED w/Video)", "Claude is the Fastest Growing Enterprise Product Ever — And It Makes It Simple to Cancel. Why Do You Make It So Hard?", "6 Reasons It’s So Hard to Sell Your Start-Up Stock as an Ex-Employee", "The 6 Threat Vectors Attacking Traditional B2B Software in 2026 (And How to Fight Back)", "A 9 Point Checklist for Sales Reps Looking at a New Role", "We’re All Expecting Too Much From Marketing These Days", "Zendesk and Anaplan: A Tale of Two Very Similar, And Very Different, $10B SaaS Acquisitions", "How SaaStr Adapted to the Pandemic with Inside Support", "Does It Seem Like Everyone in Tech is Unhappier? It’s Not Just You", "AI Wins When It Does The GTM Work … People Won’t", "Salesforce: “Actually, Things Are Now Slowing Down a Bit — At Least in Some Places”", "10+ Questions to Ask If You’re Interviewing for a VP of Sales Role", "The 250,000 Customer Club: How HubSpot and Monday.com Both Created SMB+ Empires", "Dear SaaStr: What Are The Top 3 Criteria for Hiring Your First Head of Sales?", "The Silent Churn: Why Your Best Customers Often Leave Without a Word. We Just Did.", "BVP: 52% of The Top SaaS and Cloud Leaders Are Headquartered in California", "VCs are Checking Out in Droves Right Now", "The Y Combinator Guide to Perfectly Pitching Your Seed Stage Startup with Michael Seibel", "Dear SaaStr: Do I Have to Provide a Promotion Path from SDR to AE?", "7 Things I Learned from Loren Padelford, CRO of $100m+ ARR Vertical SaaS Leader Slice", "Join CEOs & CTOs of Qualtrics, Thoughtspot, Salesforce Service Cloud, Elastic, Auth0, Zendesk at FREE SaaStr Enterprise 2021!", "Palantir’s +79% vs Bill’s -43%: The AI Divide That Defined SaaS Performance in 1H’25", "Your Buyer Isn’t Human Anymore. Neither Is Your Seller. Welcome to AI-Led Growth. With Firebolt’s President Hemanth Vedagarbha", "The Top 10 Questions to Ask a VP of Marketing in an Interview (Updated)", "The One Best, “Secret” Hack to Getting Venture Funding", "Ode to the $100,000,000 Exit", "The Average SaaS CEO is 43 at IPO (Updated)", "Need a Second Check From Your VCs? Here’s How “Reserves” Work", "What Does It Take To Be “Venture-Fundable”?", "10 of The Toughest Lessons in Building a SaaS Startup. Or Any Startup.", "2021 Was Crazy Because Growth Was Crazy. 2024 is Tough Because Growth is Tough.", "Dear SaaStr: How Do I Build a Great SDR Team?", "Acquisitions — If You Do Sell, Try to Make Sure It’s At a Local Maximum", "The Complete Guide to Vibe Coding Without a Developer: The 14 Key Lessons to Learn Before You Start", "Dear SaaStr: How Do I Deal With Investors That are Unhappy With The Company?", "5 Key Things You Need to Know About AI Data Privacy in 2025 (and Why It Matters) with Skyflow’s CPO", "The Agent Orchestration Problem: How We Wire 20+ AI Agents Together Without Losing Our Minds", "AI and Cybersecurity: How Rubrik’s Co-Founder Built a $1B+ ARR Platform While Joining the AI Revolution", "Salesforce Now Has 3+ Pricing Models for Agentforce. And Maybe Right Now, That’s The Way to Do It.", "10 Of My Favorite SaaStr Posts on Building Your First Great Sales Team", "Dear SaaStr: How Important Is a Strong LinkedIn Presence Today?", "What to Do If Your Business Decelerates", "Axios: U.S. IPOs Down 94.1%", "Dear SaaStr: Where Can I Find Commission-Only B2B Salespeople?", "The #1 Error with AI GTM Agents: Assuming They Can Do What Your Team Hasn’t Already Figured Out", "When The Team Revolts", "5 Interesting Learnings as Marin Software … Reorgs", "4 out of 800 Sales Execs Raised Their Hand. A New Low in SaaS Malaise?", "How We Went From 20+ Humans to 3 Humans + 20 AI Agents at SaaStr: More Work, Less Drama, Triple the Output", "You Can’t Hire a VP That You Don’t Love", "OpenAI’s $6B Secondary: The Largest Employee Liquidity Event in Tech History", "MCP and The Fight Over Whose Data it is: The Coming Enterprise Software Revolt", "Doubling Down: Luci Fonseca, Partner at Base10 Partners", "5 Very Good Days, and 5 Pretty Bad Days, as a SaaS CEO", "The Great Rotation: How AI/ML Crushed Traditional SaaS in Seed Investing During 1H 2025, Per AngelList", "Why Founders Leave After Acquisitions", "I’m Tired of Running My Successful Start-up After 5 Years. What Should I Do?", "The Next Big Thing in AI Compliance: What ISO 42001 Means for Your SaaS Company with Prescient Security’s COO", "Google: Actually, AI Hasn’t Hurt Search at All. In Fact, It’s Accelerating It.", "Dear SaaStr: As a Founder, What Were the Sketchiest Things a VC Asked You To Do?", "Selling Your Startup? 5 Things To Think About", "Dear SaaStr: How Do I Launch a Second Product? And Who Should Launch It?", "Who is Henry Schuck CEO and Founder ZoomInfo", "2025 Was The Year Of The AI Budget Explosion: If Your Team Didn’t Capture Any, They Failed", "The Two Biggest Mistakes I Made As a Founder-CEO", "Get the Playbooks to $100m ARR and Beyond at FREE 2021 SaaStr Scale on Dec 15!!", "Doubling Down: Nina Achadijian, Partner at Index Ventures", "Dear SaaStr: When Startup Founders “Exit”, How Often Do They Found Another Company?", "My App Stack: Meagen Eisenberg, CMO at Lacework", "The 90/10 Rule for AI Agents, A Deep Dive: When To Replace Paid SaaS Tools With a Vibe-Coded Apps. And When Not To.", "5 Reasons to Meet Your Competitor’s CEO", "We Sent 4,495 AI SDR Emails in 2 Weeks and Got The #1 Response Rate. Here’s How (It’s Not What You Think)”", "Almost Everyone’s Gotten Radically More Efficient in SaaS", "Olo vs Toast: $2B vs $25B – The Strategic Tale of Two Tech Giants", "AI Won’t Replace Sales Reps So Much as ‘Deflect’ Them: What Support’s 70% Deflection Rates Tell Us About Sales’ Future", "Data-Driven Dominance: The 6-Step Playbook That Zeni Uses to Drive 110% Revenue Growth", "Lessons Learned Rolling out AI Agents to 16,000+ SMB brands with Gorgias’ CTO", "If You Want to Get Funded – Dude, Make It Easy On Them (Updated)", "Replit’s New Release Addressed Most of The Challenges We Hit Vibe Coding. But Is ‘Prosumer’ Vibe Coding Really Ready for Commercial Apps Yet?", "Every Month is a Good Month to Raise: What 42,000+ Funding Rounds Tell Us About Raising Capital in July", "Brex and The Pros and Cons of Hubristic Fundraising", "One Reason to Maybe Run Office 365, Not Google Workplace", "Strange World Indeed, January 2026 Edition: The Massive, Unprecedented Divergence Between Public B2B and Private AI B2B", "10 Things That Make Your Investors Lose Confidence", "Gartner: SaaS Spend In the Enterprise Will Grow 17% to $195 Billion in 2023", "The #1 Worst Hire of All When You Are Just Starting To Scale: The Mediocre VP of Engineering", "Top 10 Unexpected Learnings from Scaling Wiz From $0 to The First $100M ARR with founding CRO Colin Jones and Sam Blond", "How To Do a QBR (Quarterly Business Review) Right", "Dear SaaStr: What Are Signs It’s Finally Time to Move on From An Employee?", "Top 10 SaaStr Videos of the Week: Terminus, Zendesk, Twilio, ActiveCampaign, Vanta and More!!", "SaaStrU Crosses 9,000+ Students: What We’ve Learned", "Dear SaaStr: What Are Your Top Tips to Building Your First Sales Team?", "The 5 Best Pieces of Advice I Got From My Mentors (Updated with New Video)", "Our SaaStr AI Startup Valuation Calculator Crosses 400,000 (!) Valuations. And Now We Can Tell You The Odds You’ll Get Funded. In Seconds.", "CrowdStrike Just Had Its Best Quarter Ever. So Much for AI Disrupting Cybersecurity. But The Markets Want to See … Acceleration.", "How Important Is Domain Expertise in Building a B2B Public Company? The Evidence Is … Mixed", "How The Strange and Odd World of Financial Accounting Drives Tech Acquisitions and Investments", "OpenAI vs. Anthropic: Ramp Data Shows 36% vs. 12% Penetration, But Velocity Curves Tell a Different Story", "Dear SaaStr: Will a VC Agree To Wait 5-10 Years For a Very Profitable Exit?", "How to Maximize Your Revenue: Strategies to Stop Leaving Money on the Table with Capchase’s VP Revenue", "4 Recent, Very Bad, Truly Terrible Customer Success Experiences. Just This Week.", "5 Signs You (Finally) Have Early Product-Market Fit", "5 Reasons to Actually Have Board Meetings", "What Makes a Great VP of Marketing in SaaS: A Deep Dive with Jason and Dave Gerhardt, ex-VPM of Drift", "My App Stack: Jessica Gilmartin, Chief Marketing Officer of Calendly", "64% of You Are Growing Your Sales Teams in 2024", "SaaStr APAC Will Be Back Soon!!", "Beware the “Mediocre Recycled”. The Zombie Executives of B2B."]''')
print(len(TOPICS), 'topics. First three:')
for t in TOPICS[:3]:
    print(' -', t)

450 topics. First three:
 - Don’t Fear Business Process Change. It’s (Usually) a Good Thing.
 - Do Your VPs Have a Mentor Budget?
 - Be Honest If Growth Has Slowed. Is Your Product Still Magical? It Probably … Used to Be.


### Load the model

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > GPU, then Run all again."
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{gpu_gb:.0f} GB")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
if gpu_gb >= 30:
    mode, BATCH = "bf16", 32
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda").eval()
else:
    mode, BATCH = "4bit", 8
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="cuda", quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True)).eval()
print("loaded in", mode, "| batch size", BATCH)

NVIDIA A100-SXM4-40GB 42 GB


config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded in bf16 | batch size 32


### Generation functions

In [5]:
EOS_IDS = [i for i in {tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} if i is not None]


def build_prompt(medium, topic):
    message = INSTRUCTION[medium].format(topic=topic)
    text = tok.apply_chat_template([{"role": "user", "content": message}], tokenize=False, add_generation_prompt=True)
    return text, message


@torch.inference_mode()
def generate_batch(medium, topics, seed):
    torch.manual_seed(seed)
    prompts = [build_prompt(medium, t)[0] for t in topics]
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW[medium], eos_token_id=EOS_IDS,
                         pad_token_id=tok.pad_token_id, **SAMPLING)
    results = []
    for row in out[:, enc["input_ids"].shape[1]:]:
        ids, stop = row.tolist(), "length"
        for i, x in enumerate(ids):
            if x in EOS_IDS:
                ids, stop = ids[:i], "eos"
                break
        ids = [x for x in ids if x != tok.pad_token_id]
        results.append({"text": tok.decode(ids, skip_special_tokens=True).strip(), "new_tokens": len(ids), "stop": stop})
    return results


def finished_ids():
    done = set()
    if os.path.exists(JSONL):
        with open(JSONL, encoding="utf-8") as f:
            done = {json.loads(line)["id"] for line in f if line.strip()}
    return done


def run_medium(medium):
    lo, hi = PLAN[medium]
    items = [(f"{medium}-{i:04d}", TOPICS[i], i) for i in range(lo, hi)]
    done = finished_ids()
    todo = [x for x in items if x[0] not in done]
    print(f"{medium}: {len(items)} planned, {len(done & {x[0] for x in items})} already done, {len(todo)} to do")
    t0 = time.time()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b + BATCH]
        seed = SEED + 1000 * list(PLAN).index(medium) + chunk[0][2]     # same seed whether or not we resumed
        res = generate_batch(medium, [c[1] for c in chunk], seed)
        with open(JSONL, "a", encoding="utf-8") as f:
            for (item_id, topic, _), r in zip(chunk, res):
                f.write(json.dumps({"id": item_id, "medium": medium, "topic": topic,
                                    "prompt": INSTRUCTION[medium].format(topic=topic), "seed": seed,
                                    "model": MODEL_ID, "precision": mode, **r}, ensure_ascii=False) + "\n")
        print(f"  {medium} {min(b + BATCH, len(todo))}/{len(todo)}  ({(time.time() - t0) / 60:.1f} min)", flush=True)

### Run (this is the long step)

In [6]:
for medium in ("blog", "linkedin", "talk"):
    run_medium(medium)
print("done")

blog: 450 planned, 0 already done, 450 to do


[transformers] Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  blog 32/450  (0.6 min)


[transformers] Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  blog 64/450  (1.1 min)


[transformers] Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (51 outputs left out)

[transformers] Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  talk 128/150  (2.1 min)


[transformers] Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  talk 150/150  (2.6 min)
done


### Quick check of what came out

In [7]:
rows = [json.loads(line) for line in open(JSONL, encoding="utf-8") if line.strip()]
by = collections.defaultdict(list)
for r in rows:
    by[r["medium"]].append(r)
refusal = re.compile(r"\bI (can't|cannot|can not|won't)\b|I'm not able to|as an AI", re.I)
for m, v in by.items():
    words = [len(r["text"].split()) for r in v]
    print(f"{m:9s} n={len(v):3d} median words={statistics.median(words):5.0f} min={min(words):4d} "
          f"hit token limit={sum(r['stop'] == 'length' for r in v):3d} empty={sum(w == 0 for w in words)} "
          f"possible refusals={sum(bool(refusal.search(r['text'][:200])) for r in v)}")
print()
for m, v in by.items():
    print("SAMPLE", m, "->", v[0]["text"][:350].replace("\n", " "), "\n")

blog      n=450 median words=  520 min= 292 hit token limit=101 empty=0 possible refusals=0
linkedin  n=300 median words=  241 min= 141 hit token limit=228 empty=0 possible refusals=0
talk      n=150 median words=  536 min= 353 hit token limit= 36 empty=0 possible refusals=0

SAMPLE blog -> **Don't Fear Business Process Change. It's (Usually) a Good Thing**  Hey SaaStr community,  As I've been working with and advising many growth-stage startups over the years, one thing that strikes me is how often they fear change - particularly when it comes to business processes.  And honestly? That fear is usually misplaced.  Now, before we dive  

SAMPLE linkedin -> **The #1 Killer of SaaS Growth? Not Churn. It's Rep Turnover**  Hey fellow entrepreneurs and sales leaders,  I've seen it time and time again - top-performing reps leave, taking valuable knowledge and pipeline with them. The cost is staggering (avg. $100k+ per rep). And the impact on growth? Devastating.  But there are ways to break t

### Save and download (nothing private inside: it is Llama writing)

In [8]:
sha = hashlib.sha256(open(JSONL, "rb").read()).hexdigest()
manifest = {
    "purpose": "plain-LLM negatives for the DATA 298B authorship classifier (PUK-28)",
    "model": MODEL_ID, "precision": mode, "seed": SEED, "sampling": SAMPLING, "max_new_tokens": MAX_NEW,
    "instruction_templates": INSTRUCTION, "plan": PLAN, "n_topics": len(TOPICS),
    "n_rows": len(rows), "rows_per_medium": {m: len(v) for m, v in by.items()},
    "jsonl_sha256": sha, "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__, "python": platform.python_version(),
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "note": "Same prompt wording and sampling as the Base Llama baseline EXP-20261001-010, but the HF fp16/4-bit weights, "
            "not the Ollama Q4_K_M build used for that baseline, so wording will differ slightly.",
}
json.dump(manifest, open(f"{OUT_DIR}/manifest.json", "w"), indent=2)
bundle = "/content/llm_negatives_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(JSONL, "llm_negatives.jsonl")
    z.write(f"{OUT_DIR}/manifest.json", "manifest.json")
print("rows:", len(rows), "| sha256:", sha[:16], "...")
from google.colab import files
files.download(bundle)

rows: 900 | sha256: 6c6b2fbaef920f99 ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Optional: also keep a copy on Google Drive
Off by default, so "Run all" never asks you to sign in. Set `SAVE_TO_DRIVE = True` and run this cell by itself if you want it.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/298B_colab", exist_ok=True)
    shutil.copy(bundle, "/content/drive/MyDrive/298B_colab/llm_negatives_bundle.zip")
    print("copied to Drive: 298B_colab/llm_negatives_bundle.zip")
else:
    print("Drive copy skipped (SAVE_TO_DRIVE is False).")

### After this
Download `llm_negatives_bundle.zip`, tell Claude, and the classifier is rerun with these negatives
(`python run_pipeline.py --llm-negatives llm_negatives.jsonl --out-dir outputs_colab`).